In [1]:
# load dungeon data

import json
import random
from pathlib import Path

data_path = Path("../datasets/dungeon_10k_4_8_3_5_mkr.jsonl")
with open(data_path) as f:
    data = [json.loads(line) for line in f]

# strip _id fields
for entry in data:
    if "_id" in entry:
        del entry["_id"]

# Split into train/eval sets
random.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
eval_data = data[split_idx:]

print(f"Train set: {len(train_data)} records")
print(f"Eval set: {len(eval_data)} records")

# print first data point
print(json.dumps(data[0], indent=2))

Train set: 8000 records
Eval set: 2000 records
{
  "door": 3,
  "key_color": "red",
  "corridor": [
    {
      "monsters": [
        "dragon",
        "goblin"
      ],
      "door_no": 3,
      "blue_key": "gemstones",
      "green_key": "artifacts",
      "red_key": "diamonds"
    },
    {
      "door_no": 1,
      "red_key": "gemstones",
      "green_key": "artifacts",
      "blue_key": "artifacts"
    },
    {
      "door_no": 6,
      "green_key": "gemstones",
      "red_key": "gold",
      "blue_key": "gemstones"
    },
    {
      "monsters": [
        "troll"
      ],
      "door_no": 4,
      "blue_key": "artifacts",
      "red_key": "gemstones",
      "green_key": "diamonds"
    },
    {
      "monsters": [
        "orc",
        "wolf"
      ],
      "door_no": 2,
      "blue_key": "diamonds",
      "green_key": "diamonds",
      "red_key": "gemstones"
    },
    {
      "monsters": [
        "goblin",
        "orc"
      ],
      "door_no": 0,
      "green_key": "spellbook

In [4]:
from origami import ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.config import DataConfig
from origami.training import TableLogCallback, accuracy

config = OrigamiConfig(
    data=DataConfig(
        infer_schema=True,
        numeric_mode="disabled",
    ),
    model=ModelConfig(
        backbone="transformer",
        kvpe_pooling="sum",
        d_model=128,
        n_heads=8,
        n_layers=6,
        d_ff=784,
        dropout=0.0,
    ),
    training=TrainingConfig(
        shuffle_keys=False,
        batch_size=64,
        warmup_steps=1000,
        learning_rate=1e-3,
        eval_strategy="epoch",
        eval_epochs=5,
        eval_metrics={"acc": accuracy},
        eval_sample_size=100,
        eval_on_train=True,
        target_key="treasure",
        target_loss_weight=1.0,
        constrain_grammar=True,
        constrain_schema=True,
    ),
    device="mps",
)

pipeline = OrigamiPipeline(config)
pipeline.fit(
    train_data,
    eval_data=eval_data,
    callbacks=[TableLogCallback(print_every=100)],
    epochs=100,
    verbose=True,
)

Vocabulary size: 40
Derived schema:
{
  "type": "object",
  "properties": {
    "corridor": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "blue_key": {
            "type": "string",
            "enum": [
              "artifacts",
              "diamonds",
              "gemstones",
              "gold",
              "spellbooks"
            ]
          },
          "door_no": {
            "type": "integer",
            "enum": [
              0,
              1,
              2,
              3,
              4,
              "... + 3 more"
            ],
            "minimum": 0,
            "maximum": 7
          },
          "green_key": {
            "type": "string",
            "enum": [
              "artifacts",
              "diamonds",
              "gemstones",
              "gold",
              "spellbooks"
            ]
          },
          "monsters": {
            "type": "array",
            "items": {
      

OrigamiPipeline(numeric_mode='disabled', fitted)

In [ ]:
# pipeline.save("dungeon_pipeline.pt")

In [ ]:
from origami import OrigamiPipeline

# pipeline = OrigamiPipeline.load("dungeon_pipeline.pt")

In [5]:
from origami.training import accuracy

pipeline.evaluate(eval_data, metrics={"acc": accuracy})

{'loss': 0.714148413567316, 'acc': 0.999}

In [7]:
doc = pipeline.generate(1)[0]

print(json.dumps(doc, indent=2))

{
  "door": 1,
  "key_color": "red",
  "corridor": [
    {
      "door_no": 1,
      "blue_key": "artifacts",
      "green_key": "gold",
      "red_key": "artifacts"
    },
    {
      "door_no": 2,
      "red_key": "gold",
      "green_key": "artifacts",
      "monsters": [
        "goblin",
        "wolf"
      ],
      "blue_key": "artifacts"
    },
    {
      "monsters": [
        "troll",
        "orc"
      ],
      "door_no": 4,
      "red_key": "artifacts",
      "blue_key": "spellbooks",
      "green_key": "artifacts"
    },
    {
      "monsters": [
        "dragon",
        "wolf"
      ],
      "door_no": 5,
      "red_key": "spellbooks",
      "green_key": "gold",
      "blue_key": "artifacts"
    },
    {
      "monsters": [
        "wolf",
        "dragon"
      ],
      "door_no": 6,
      "red_key": "gold",
      "green_key": "gold",
      "blue_key": "gemstones"
    },
    {
      "monsters": [
        "goblin",
        "orc"
      ],
      "door_no": 3,
      "green